# 05 — Controlled prompt experiments

We compare three handwritten transcription instructions using the same trained adapter and validation regions. Only the handwritten prompt changes; classification and other region prompts remain fixed.

The original-prompt evaluation comes from notebook 04. Completed results for the two additional variants are also reused when their configurations match. Missing or partial variants require new or resumed inference, potentially taking two full validation passes.

The default adapter remains `multitask_smoke`, matching the 64-region training run. Its prompt experiments are exploratory checks. Use `gold_multitask` only after completing full training and the corresponding notebook 04 evaluation.

In [ ]:
import json
import sys
from pathlib import Path

import torch

CWD = Path.cwd().resolve()
CANDIDATES = [candidate for parent in (CWD, *CWD.parents) for candidate in (parent, parent / "src")]
PACKAGE = next(
    (
        candidate
        for candidate in CANDIDATES
        if (candidate / "rukopys" / "evaluate.py").is_file() and (candidate / "work").is_dir()
    ),
    None,
)

if PACKAGE is None:
    raise FileNotFoundError(
        "Open the notebook from the project directory or one of its subdirectories. "
        "Place the rukopys package inside the existing src directory, next to work."
    )

if str(PACKAGE) not in sys.path:
    sys.path.insert(0, str(PACKAGE))

from rukopys.config import CLASS_PROMPT, PROMPTS
from rukopys.evaluate import evaluate
from rukopys.models import load_qwen
from rukopys.utils import atomic_json, digest, file_digest, read_json


def percent(value):
    """Format a metric as a percentage, preserving missing values."""
    return "N/A" if value is None else f"{value:.2%}"


ROOT = (PACKAGE / "work").resolve()
RUN_NAME = "multitask_smoke"
REUSE_COMPLETED_RESULTS = True
RUN_DIR = ROOT / "runs" / RUN_NAME
REFERENCE_PATH = ROOT / "outputs" / f"{RUN_NAME}_validation.json"

for path in (
    REFERENCE_PATH,
    ROOT / "project.json",
    ROOT / "data" / "validation.json",
    RUN_DIR / "latest.json",
):
    if not path.is_file():
        raise FileNotFoundError(f"Missing required artifact: {path.relative_to(PACKAGE)}")

reference = read_json(REFERENCE_PATH)
validation = read_json(ROOT / "data" / "validation.json")
project = read_json(ROOT / "project.json")

if not reference.get("complete") or "summary" not in reference:
    raise ValueError("Complete the matching notebook 04 evaluation first.")

if not validation or reference["config"]["manifest"] != digest(validation):
    raise ValueError("The validation manifest is empty or differs from notebook 04.")

if [row["id"] for row in reference["results"]] != [row["id"] for row in validation]:
    raise ValueError("The reference evaluation does not cover the same regions.")

if reference["config"]["prompts"] != PROMPTS or reference["config"]["class_prompt"] != CLASS_PROMPT:
    raise ValueError("Current prompts differ from the reference evaluation.")

CHECKPOINT = Path(read_json(RUN_DIR / "latest.json")["checkpoint"]).resolve()

if CHECKPOINT.parent != RUN_DIR.resolve():
    raise ValueError("The checkpoint must belong to the selected run directory.")

for name in (
    "COMPLETE",
    "adapter_model.safetensors",
    "adapter_config.json",
    "run_config.json",
):
    if not (CHECKPOINT / name).is_file():
        raise FileNotFoundError(f"Incomplete checkpoint: missing {name}.")

reference_protocol = reference["config"]["protocol"]
MAX_PIXELS = reference_protocol["max_pixels"]
MAX_NEW_TOKENS = reference["config"]["max_tokens"]
ADAPTER_SHA256 = file_digest(CHECKPOINT / "adapter_model.safetensors")
training_config = read_json(CHECKPOINT / "run_config.json")

if (
    reference_protocol["adapter_sha256"] != ADAPTER_SHA256
    or reference_protocol["model_id"] != project["model_id"]
    or reference_protocol["model_revision"] != project.get("model_revision")
    or reference_protocol["quantized"] is not True
    or training_config["model_revision"] != project["model_revision"]
    or training_config["protocol"]["max_pixels"] != MAX_PIXELS
):
    raise ValueError("The checkpoint or model settings differ from notebook 04.")

print("Python:", sys.version.split()[0])
print("Source directory:", PACKAGE.name)
print("Checkpoint:", CHECKPOINT.relative_to(PACKAGE))
print("Validation regions:", len(validation))

Python: 3.12.5
Source directory: src
Checkpoint: work\runs\multitask_smoke\checkpoint_000031
Validation regions: 5082


In [ ]:
variants = {
    "original": dict(PROMPTS),
    "short": {
        **PROMPTS,
        "handwritten": "Read all Ukrainian handwriting in this crop. Copy exactly, preserving punctuation and incomplete words. Return only the transcription.",
    },
    "uk": {
        **PROMPTS,
        "handwritten": "Розпізнай увесь український рукописний текст у цій області. Збережи оригінальне написання, пунктуацію та незавершені слова. Не виправляй і не дописуй текст. Поверни лише транскрипцію.",
    },
}

EXPERIMENT_DIR = ROOT / "outputs" / f"{RUN_NAME}_prompt_experiments" / ADAPTER_SHA256
experiment_results = {"original": reference["summary"]}
PENDING_VARIANTS = []

for name in ("short", "uk"):
    output = EXPERIMENT_DIR / f"{name}.json"

    if not output.is_file():
        PENDING_VARIANTS.append(name)
        continue

    saved = read_json(output)
    expected_config = {
        "manifest": digest(validation),
        "protocol": reference_protocol,
        "prompts": variants[name],
        "class_prompt": CLASS_PROMPT,
        "max_tokens": MAX_NEW_TOKENS,
    }

    if saved.get("config") != expected_config:
        raise ValueError(
            f"Saved prompt variant {name!r} uses a different configuration. "
            "Keep its output and choose a separate experiment directory."
        )

    saved_results = saved.get("results", [])

    if [row["id"] for row in saved_results] != [
        row["id"] for row in validation[: len(saved_results)]
    ]:
        raise ValueError(f"Saved region order differs for variant {name!r}.")

    complete = (
        saved.get("complete", False)
        and len(saved_results) == len(validation)
        and "summary" in saved
    )

    if REUSE_COMPLETED_RESULTS and complete:
        experiment_results[name] = saved["summary"]
    else:
        PENDING_VARIANTS.append(name)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Original evaluation reused:", REFERENCE_PATH.relative_to(PACKAGE))
print("Experiment directory:", EXPERIMENT_DIR.relative_to(PACKAGE))
print("Variants requiring evaluation:", PENDING_VARIANTS)

PyTorch: 2.7.1+cu126
CUDA available: True
Original evaluation reused: work\outputs\multitask_smoke_validation.json
Experiment directory: work\outputs\multitask_smoke_prompt_experiments\565cc647b703b93c2e4257a0aa7b9a4d95a7326915e174873d5d045936228e06
Variants requiring evaluation: []


## Prompt variants and model loading

We keep the original instruction as the reference and compare it with a shorter English instruction and a Ukrainian instruction. The Ukrainian prompt intentionally remains in Ukrainian because instruction language is part of the experiment; explanations and comments remain in English.

Reference text is not used to repair predictions. Validation labels provide metrics, and selecting a prompt on validation is model selection rather than an independent test result.

Qwen loads only if at least one variant needs evaluation. The current model and runtime protocol must then match notebook 04 exactly. When all results are complete, we reuse their recorded protocol without requiring GPU inference or checking current runtime versions.

In [3]:
if PENDING_VARIANTS:
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA-enabled kernel to run or resume inference.")

    print("GPU:", torch.cuda.get_device_name(0))
    model, processor, protocol = load_qwen(
        ROOT,
        adapter=CHECKPOINT,
        quantized=True,
        max_pixels=MAX_PIXELS,
    )

    if protocol != reference_protocol:
        raise ValueError(
            "The current model, adapter, or runtime differs from notebook 04. "
            "Restore the reference environment and settings before continuing."
        )
else:
    protocol = reference_protocol
    print("All prompt evaluations are complete; model loading is skipped.")

print(json.dumps(protocol, ensure_ascii=False, indent=2))

All prompt evaluations are complete; model loading is skipped.
{
  "model_id": "Qwen/Qwen3-VL-4B-Instruct",
  "model_revision": "ebb281ec70b05090aa6165b016eac8ec08e71b17",
  "quantized": true,
  "dtype": "torch.bfloat16",
  "max_pixels": 262144,
  "adapter_sha256": "565cc647b703b93c2e4257a0aa7b9a4d95a7326915e174873d5d045936228e06",
  "packages": {
    "torch": "2.7.1+cu126",
    "transformers": "4.57.6",
    "peft": "0.21.2",
    "bitsandbytes": "0.49.0",
    "pillow": "11.1.0"
  }
}


## Evaluate each variant

We reuse each complete matching variant independently. Missing outputs are created, while partial evaluations resume after their saved regions. An interrupted experiment therefore does not require rerunning completed variants.

New inference evaluates all region types even though only the handwritten prompt changes. Handwritten metrics are reported separately to isolate that change. Raw CER with reference boxes is not the full Kaggle score.

The comparison file is created when absent and kept unchanged when it already matches. A matching partial comparison may be extended with additional variant results. Conflicting existing results or settings stop execution instead of being replaced.

In [ ]:
for name in ("short", "uk"):
    if name not in PENDING_VARIANTS:
        print(f"Reusing completed prompt variant: {name}")
        continue

    print(f"Evaluating prompt variant: {name}")
    experiment_results[name] = evaluate(
        ROOT,
        model,
        processor,
        protocol,
        EXPERIMENT_DIR / f"{name}.json",
        prompts=variants[name],
        max_tokens=MAX_NEW_TOKENS,
    )

    # Mark completed work so rerunning this cell can reuse it in the same kernel.
    PENDING_VARIANTS.remove(name)

comparison = {
    "run_name": RUN_NAME,
    "checkpoint": str(CHECKPOINT),
    "protocol": protocol,
    "manifest": reference["config"]["manifest"],
    "max_tokens": MAX_NEW_TOKENS,
    "variants": variants,
    "results": experiment_results,
    "scope": (
        "Validation prompt selection; reference boxes; " "raw CER without official normalization"
    ),
}
comparison_path = EXPERIMENT_DIR / "comparison.json"

if comparison_path.is_file():
    previous_comparison = read_json(comparison_path)
    previous_settings = {
        key: value for key, value in previous_comparison.items() if key != "results"
    }
    current_settings = {key: value for key, value in comparison.items() if key != "results"}

    if previous_settings != current_settings or any(
        name not in experiment_results or experiment_results[name] != result
        for name, result in previous_comparison.get("results", {}).items()
    ):
        raise ValueError(
            "The existing comparison conflicts with the current experiment. "
            "Keep it and select a separate comparison filename."
        )

    if previous_comparison == comparison:
        print("Existing comparison retained:", comparison_path.relative_to(PACKAGE))
    else:
        atomic_json(comparison_path, comparison)
        print("Partial comparison completed:", comparison_path.relative_to(PACKAGE))
else:
    atomic_json(comparison_path, comparison)
    print("Comparison saved:", comparison_path.relative_to(PACKAGE))

print("Prompt experiments completed.")

Reusing completed prompt variant: short
Reusing completed prompt variant: uk
Existing comparison retained: work\outputs\multitask_smoke_prompt_experiments\565cc647b703b93c2e4257a0aa7b9a4d95a7326915e174873d5d045936228e06\comparison.json
Prompt experiments completed.


## Compare results

We compare handwritten oracle-type CER to isolate the transcription instruction. Overall routed CER also captures errors introduced by predicted-type routing.

Corpus CER and mean region CER weight examples differently, so the preferred prompt can depend on the metric. A conclusion should consider the measured values and reviewed examples rather than assume that every metric improves.

In [ ]:
print(
    "Variant | Handwritten corpus CER | " "Handwritten mean region CER | Overall routed corpus CER"
)

for name in ("original", "short", "uk"):
    summary = experiment_results[name]
    handwritten = summary["by_type"]["handwritten"]

    print(
        " | ".join(
            [
                name,
                percent(handwritten.get("corpus_raw_cer")),
                percent(handwritten.get("mean_region_raw_cer")),
                percent(summary["routed"].get("corpus_raw_cer")),
            ]
        )
    )

print("Detailed comparison:", comparison_path.relative_to(PACKAGE))

Variant | Handwritten corpus CER | Handwritten mean region CER | Overall routed corpus CER
original | 86.44% | 90.57% | 86.84%
short | 88.91% | 79.61% | 88.25%
uk | 109.83% | 99.83% | 106.87%
Detailed comparison: work\outputs\multitask_smoke_prompt_experiments\565cc647b703b93c2e4257a0aa7b9a4d95a7326915e174873d5d045936228e06\comparison.json


## Interpretation and later training ablations

- Best handwritten instruction on validation: [to be completed after review]
- Changes in punctuation, omitted words, and invented text: [to be completed after review]
- Routed transcription observations: [to be completed after review]

These prompt experiments do not train a new adapter. For training ablations, we use separate fresh kernels and new run names in notebook 03, then evaluate on the same validation manifest in notebook 04. Only one factor changes at a time.

| Run | Learning rate | Rank | Base quantization | Purpose |
|---|---:|---:|---|---|
| gold_multitask | 1e-4 | 8 | NF4 | Full gold reference |
| gold_lr5e5 | 5e-5 | 8 | NF4 | Learning rate ablation |
| gold_r16 | 1e-4 | 16 | NF4 | Adapter rank ablation |
| gold_lora | 1e-4 | 8 | None | LoRA comparison if memory permits |

An unquantized LoRA experiment requires a separate configuration in notebook 03: `quantized=False` in model loading and an update to its quantization checks. The current notebook 03 is configured for QLoRA. Memory feasibility should be checked first; successful quantized inference does not guarantee that unquantized training fits on an 11 GiB GPU. A failed run is recorded as a feasibility result rather than a recognition score.

Silver-to-gold training and the external Rukopys-OCR benchmark remain separate experiments requiring their own manifests, filtering, and overlap checks. Results from an older 159-crop evaluation are not transferred to this validation set.

Before using a selected prompt in notebook 06, we explicitly update its transcription configuration and record the chosen prompt with the submission experiment.